# 🐾 댕댕맵 (DaengDaeng Map)
## 전국 반려견 동반 관광 맞춤 추천 플랫폼
> **프로젝트**: 2026 한국관광공사 관광데이터 활용 공모전  
> **팀**: 빅데이터경영학과 3학년 2인팀  
> **데이터 출처**: 한국관광공사 TourAPI + 전국 반려동물 동반 가능 문화시설 위치 데이터(2023)

---
### 📋 서비스 개요
견주가 강아지 정보(이름, 견종, 크기)를 등록하면 전국 반려동물 동반 가능 장소를 지도에서 탐색하고  
AI가 맞춤 코스를 추천해주는 서비스

### 📋 노트북 구성
| 단계 | 내용 |
|---|---|
| 1 | 라이브러리 import 및 환경 설정 |
| 2 | 데이터 수집 (TourAPI + 문화시설 CSV 통합) |
| 3 | EDA (탐색적 데이터 분석) |
| 4 | 전처리 |
| 5 | 분석 및 시각화 |
| 6 | 데이터 저장 |

## 1. 라이브러리 import 및 환경 설정

In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import time, os
from datetime import datetime
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv("../.env")

# 한글 폰트 설정 (Windows)
font_path = r"C:\Windows\Fonts\malgun.ttf"
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    prop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = prop.get_name()
else:
    print("⚠️ 폰트를 찾을 수 없습니다:", font_path)

OUTPUT_DIR = "./data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ 환경 설정 완료")
print(f"   pandas  : {pd.__version__}")
print(f"   실행 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


✅ 환경 설정 완료
   pandas  : 3.0.1
   실행 시간: 2026-05-26 19:02:47


## 2. 데이터 수집
> **데이터 소스**
> - `TourAPI` : 한국관광공사 반려동물 동반여행 서비스 (areaBasedList2 + detailPetTour2)
> - `문화시설 CSV` : 전국 반려동물 동반 가능 문화시설 위치 데이터(2023)

In [ ]:
SERVICE_KEY = os.environ.get("TOUR_API_KEY")
BASE_URL    = "http://apis.data.go.kr/B551011/KorPetTourService2"

if not SERVICE_KEY:
    print("TOUR_API_KEY 환경변수가 설정되지 않았습니다.")
else:
    print(f"TOUR_API_KEY 로드됨 - {len(SERVICE_KEY)}글자")

# TourAPI areaCode 매핑 — 실제 API 호출로 샘플 주소 검증 완료
# 검증 결과: 코드 35~38은 전북/전남/경북/경남 순이 아니라 경북/경남/전북/전남 순임
AREA_CODES = {
    "1":  "서울",
    "2":  "인천",
    "3":  "대전",
    "4":  "대구",
    "5":  "광주",
    "6":  "부산",
    "7":  "울산",
    "8":  "세종",
    "31": "경기",
    "32": "강원",
    "33": "충북",
    "34": "충남",
    "35": "경북",   # 검증: addr1 = "경상북도 경주시..."
    "36": "경남",   # 검증: addr1 = "경상남도 밀양시..."
    "37": "전북",   # 검증: addr1 = "전북특별자치도 순창군..."
    "38": "전남",   # 검증: addr1 = "전라남도 해남군..."
    "39": "제주",
}

def fetch_pet_tour(area_code, num_of_rows=100, page_no=1):
    params = {
        "serviceKey" : SERVICE_KEY,
        "numOfRows"  : num_of_rows,
        "pageNo"     : page_no,
        "MobileOS"   : "ETC",
        "MobileApp"  : "DaengDaengMap",
        "_type"      : "json",
        "areaCode"   : area_code,
    }
    try:
        res        = requests.get(f"{BASE_URL}/areaBasedList2", params=params, timeout=10)
        res.raise_for_status()
        body       = res.json().get("response", {}).get("body", {})
        items_raw  = body.get("items", {})
        # 결과 없을 때 API가 items: "" (빈 문자열) 반환하는 경우 대비
        items      = items_raw.get("item", []) if isinstance(items_raw, dict) else []
        total      = body.get("totalCount", 0)
    except Exception as e:
        print(f"  ⚠️ areaCode {area_code} 수집 실패: {e}")
        items, total = [], 0
    return items, total

# STEP 1: 시도별 건수 사전 확인
print("시도별 건수 확인:")
print("-" * 40)
verified_codes = {}
for area_code, area_name in AREA_CODES.items():
    items, total = fetch_pet_tour(area_code, num_of_rows=1)
    sample_addr = items[0].get("addr1", "") if items else ""
    verified_codes[area_code] = total
    print(f"  코드 {area_code:>2} ({area_name}): {total}건 | {sample_addr[:35]}")
    time.sleep(0.2)

print(f"\n전국 합계: {sum(verified_codes.values())}건")

In [ ]:
# STEP 2: 위 검증 결과 확인 후 실행 — 전국 전체 수집
# (샘플주소와 시도명이 불일치하는 코드가 있으면 AREA_CODES를 수정한 뒤 실행)
all_items = []
print("전국 반려동물 동반 가능 장소 전체 수집 시작...")
for area_code, area_name in AREA_CODES.items():
    total = verified_codes.get(area_code, 0)
    if total == 0:
        continue
    pages = (total // 100) + 1
    for page in range(1, pages + 1):
        items, _ = fetch_pet_tour(area_code, num_of_rows=100, page_no=page)
        for item in items:
            item["area_name"] = area_name
        all_items.extend(items)
        time.sleep(0.3)
    print(f"  {area_name}: {total}건 수집 완료")

tour_df = pd.DataFrame(all_items)
print(f"\n✅ 목록 수집 완료: {len(tour_df)}행 x {len(tour_df.columns)}열")


In [ ]:
# 반려동물 상세정보 수집 (detailPetTour2)
# ※ 전국 수집 후 API 할당량 소진 시 자동 중단 → TourAPI 목록 데이터만으로 진행
print("펫 상세정보 수집 중...")
detail_list = []
fail_count  = 0
SLEEP_BASE  = 1.0          # 요청 간 기본 대기 (초)
RETRY_WAIT  = 5.0          # 429 발생 시 대기 (초)
MAX_RETRIES = 2            # 재시도 횟수
ABORT_LIMIT = 3            # 연속 N건 모두 429 → 할당량 소진으로 판단하고 전체 중단

consec_429 = 0  # 연속 429 카운터

for i, row in tour_df.iterrows():

    # 할당량 소진 감지 → 남은 항목 빈 dict으로 채우고 스킵
    if consec_429 >= ABORT_LIMIT:
        detail_list.append({"contentid": row["contentid"]})
        continue

    detail      = {}
    item_was_429 = False

    for attempt in range(MAX_RETRIES):
        try:
            res = requests.get(f"{BASE_URL}/detailPetTour2", params={
                "serviceKey" : SERVICE_KEY,
                "MobileOS"   : "ETC",
                "MobileApp"  : "DaengDaengMap",
                "_type"      : "json",
                "contentId"  : row["contentid"],
            }, timeout=10)

            if res.status_code == 429:
                item_was_429 = True
                if attempt < MAX_RETRIES - 1:
                    wait = RETRY_WAIT * (attempt + 1)
                    print(f"  ⏳ Rate limit ({row['contentid']}), {wait:.0f}s 대기 후 재시도 ({attempt+1}/{MAX_RETRIES})...")
                    time.sleep(wait)
                    continue
                fail_count += 1
                break

            res.raise_for_status()
            body   = res.json().get("response", {}).get("body", {})
            items  = body.get("items")
            detail = items.get("item", [{}])[0] if isinstance(items, dict) else {}
            item_was_429 = False
            break

        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                fail_count += 1
                item_was_429 = False

    # 연속 429 카운터 갱신
    if item_was_429:
        consec_429 += 1
        if consec_429 == ABORT_LIMIT:
            remaining = len(tour_df) - i - 1
            print(f"\n⛔ API 일일 할당량 소진 감지 (연속 {ABORT_LIMIT}건 429)")
            print(f"   나머지 {remaining}건 상세수집 생략 → TourAPI 항목은 acmpyTypeCd 등 '정보없음' 처리됩니다.")
            print(f"   CSV 데이터(다수)에는 펫 정보 완전 보유. 다음 단계 정상 진행 가능합니다.")
    else:
        consec_429 = 0

    detail["contentid"] = row["contentid"]
    detail_list.append(detail)
    time.sleep(SLEEP_BASE)

    if (i + 1) % 50 == 0 and consec_429 < ABORT_LIMIT:
        print(f"  {i+1}/{len(tour_df)}건 완료")

detail_df  = pd.DataFrame(detail_list)
has_detail = detail_df[detail_df.columns.drop("contentid")].notna().any(axis=1)
print(f"\n펫 상세정보 있는 장소: {has_detail.sum()}건 / 전체 {len(detail_df)}건 (실패: {fail_count}건)")

# 목록 + 상세정보 합치기
tour_df = pd.merge(tour_df, detail_df, on="contentid", how="left")
print(f"✅ TourAPI 수집 완료: {len(tour_df)}행 × {len(tour_df.columns)}열")

In [ ]:
# 문화시설 CSV 로드 및 전처리 (전국)
csv_raw = pd.read_csv(
    "전국_반려동물_동반_가능_문화시설_위치_데이터_2023_.csv",
    encoding="utf-8"
)

# 시도명 → 단축명 매핑 (2023·2024년 행정구역 개칭 반영)
PROV_MAP = {
    "서울특별시": "서울", "부산광역시": "부산", "대구광역시": "대구",
    "인천광역시": "인천", "광주광역시": "광주", "대전광역시": "대전",
    "울산광역시": "울산", "세종특별자치시": "세종", "경기도": "경기",
    "강원특별자치도": "강원", "강원도": "강원",      # 2023년 개칭
    "충청북도": "충북", "충청남도": "충남",
    "전북특별자치도": "전북", "전라북도": "전북",    # 2024년 개칭
    "전라남도": "전남", "경상북도": "경북",
    "경상남도": "경남", "제주특별자치도": "제주",
}

# 전국 전체 데이터 사용 (카테고리 무관)
csv_all = csv_raw.copy()
print(f"전국 로드 완료: {len(csv_all)}건")
print(csv_all["CTPRVN_NM"].value_counts().head(10))

# 카테고리 매핑
cat_map = {
    "카페": "음식점", "식당": "음식점", "레스토랑": "음식점",
    "박물관": "문화시설", "미술관": "문화시설", "문예회관": "문화시설",
    "도서관": "문화시설", "전시관": "문화시설",
    "여행지": "관광지", "공원": "관광지", "해수욕장": "관광지",
    "산": "관광지", "캠핑장": "관광지",
    "펜션": "숙박", "호텔": "숙박", "모텔": "숙박", "리조트": "숙박",
    "애견호텔": "숙박",
}

# 컬럼 표준화
csv_df = pd.DataFrame({
    "title"           : csv_all["FCLTY_NM"].values,
    "addr1"           : csv_all["RDNMADR_NM"].values,
    "province"        : csv_all["CTPRVN_NM"].map(PROV_MAP).fillna(csv_all["CTPRVN_NM"]).values,
    "gu_name"         : csv_all["SIGNGU_NM"].values,
    "mapx"            : pd.to_numeric(csv_all["LC_LO"], errors="coerce").values,
    "mapy"            : pd.to_numeric(csv_all["LC_LA"], errors="coerce").values,
    "tel"             : csv_all["TEL_NO"].astype(str).values,
    "acmpyTypeCd"     : csv_all["IN_PLACE_ACP_POSBL_AT"].map({"Y": "실내 동반가능", "N": "실외만 가능"}).values,
    "acmpyPsblCpam"   : csv_all["ENTRN_POSBL_PET_SIZE_VALUE"].values,
    "acmpyNeedMtr"    : csv_all["PET_LMTT_MTR_CN"].values,
    "etcAcmpyInfo"    : csv_all["PET_INFO_CN"].values,
    "indoor_possible" : (csv_all["IN_PLACE_ACP_POSBL_AT"] == "Y").values,
    "outdoor_possible": (csv_all["OUT_PLACE_ACP_POSBL_AT"] == "Y").values,
    "homepage"        : csv_all["HMPG_URL"].values,
    "opertime"        : csv_all["OPER_TIME"].values,
    "content_type_name": csv_all["CTGRY_THREE_NM"].map(cat_map).fillna("기타").values,
    "data_source"     : "문화시설CSV",
})

print(f"\n✅ CSV 전처리 완료: {len(csv_df)}행 x {len(csv_df.columns)}열")
print("\n[시도별 현황 상위 10]")
print(csv_df["province"].value_counts().head(10))

In [ ]:
import re

# TourAPI에 CSV 전용 컬럼 추가
tour_df["indoor_possible"]  = None
tour_df["outdoor_possible"] = None
tour_df["homepage"]         = None
tour_df["opertime"]         = None
tour_df["data_source"]      = "TourAPI"

# 공통 컬럼 정의 (province 추가)
common_cols = ["title", "addr1", "province", "gu_name", "mapx", "mapy", "tel",
               "acmpyTypeCd", "acmpyPsblCpam", "acmpyNeedMtr", "etcAcmpyInfo",
               "indoor_possible", "outdoor_possible",
               "homepage", "opertime", "content_type_name", "data_source"]

# addr1에서 시도명(province)·구군명(gu_name) 추출 함수
def province_from_addr(addr):
    if pd.isna(addr): return None
    for full, short in PROV_MAP.items():
        if full in str(addr): return short
    return None

def gu_from_addr(addr):
    if pd.isna(addr): return None
    s = str(addr)
    # 구/군 우선 (가장 구체적)
    m = re.search(r'([가-힣]+[구군])\b', s)
    if m: return m.group(1)
    # 도 소속 시 (eg: 춘천시, 수원시 등) — 광역시·특별시명은 PROV_MAP에서 처리되므로 제외
    m = re.search(r'([가-힣]{2,5}시)\b', s)
    if m and not any(m.group(1) in k for k in PROV_MAP):
        return m.group(1)
    return None

# TourAPI: sigungucode 대신 addr1 기반으로 province/gu_name 설정
tour_df["province"] = tour_df["addr1"].apply(province_from_addr).fillna(tour_df["area_name"])
tour_df["gu_name"]  = tour_df["addr1"].apply(gu_from_addr).fillna("정보없음")

# contenttypeid → content_type_name
type_map = {"12": "관광지", "14": "문화시설", "32": "숙박",
            "39": "음식점", "28": "레포츠",  "38": "쇼핑"}
tour_df["content_type_name"] = tour_df["contenttypeid"].astype(str).map(type_map).fillna("기타")

# 공통 컬럼만 선택
tour_standard = tour_df[common_cols].copy()
csv_standard  = csv_df[common_cols].copy()

# 통합 후 중복 제거 (장소명 + 자치구 기준)
raw_df = pd.concat([tour_standard, csv_standard], ignore_index=True)
before = len(raw_df)
raw_df = raw_df.drop_duplicates(subset=["title", "gu_name"], keep="first")

print(f"중복 제거: {before - len(raw_df)}건 제거")
print(f"\n✅ 통합 완료 (raw_df): {len(raw_df)}건")
print(f"  - TourAPI     : {(raw_df['data_source']=='TourAPI').sum()}건")
print(f"  - 문화시설CSV : {(raw_df['data_source']=='문화시설CSV').sum()}건")
print(f"\n[시도별 현황 상위 10]")
print(raw_df["province"].value_counts().head(10))
print(f"\n[카테고리별 현황]")
print(raw_df["content_type_name"].value_counts())

## 3. EDA (탐색적 데이터 분석)
> 전처리 전, 통합 원본 데이터의 구조와 품질을 파악합니다.


In [6]:
print("=" * 55)
print("📋 데이터 기본 정보")
print("=" * 55)
print(f"데이터 크기  : {raw_df.shape[0]}행 × {raw_df.shape[1]}열")
print(f"\n컬럼 목록:")
for i, col in enumerate(raw_df.columns, 1):
    print(f"  {i:2}. {col}")


📋 데이터 기본 정보
데이터 크기  : 461행 × 16열

컬럼 목록:
   1. title
   2. addr1
   3. gu_name
   4. mapx
   5. mapy
   6. tel
   7. acmpyTypeCd
   8. acmpyPsblCpam
   9. acmpyNeedMtr
  10. etcAcmpyInfo
  11. indoor_possible
  12. outdoor_possible
  13. homepage
  14. opertime
  15. content_type_name
  16. data_source


In [7]:
print("=" * 55)
print("🔍 결측값 현황 (NaN + 빈 문자열 포함)")
print("=" * 55)
for col in raw_df.columns:
    null_cnt  = raw_df[col].isna().sum()
    empty_cnt = (raw_df[col] == "").sum() if raw_df[col].dtype == object else 0
    total     = null_cnt + empty_cnt
    if total > 0:
        print(f"{col:<25} NaN:{null_cnt:3} | 빈문자열:{empty_cnt:3} | 합계:{total:3}")
print(f"\n결측 없는 컬럼: {(raw_df.isnull().sum() == 0).sum()}개")


🔍 결측값 현황 (NaN + 빈 문자열 포함)
addr1                     NaN:  8 | 빈문자열:  0 | 합계:  8
tel                       NaN: 12 | 빈문자열:  0 | 합계: 12
acmpyTypeCd               NaN:  1 | 빈문자열:  0 | 합계:  1
acmpyPsblCpam             NaN:  1 | 빈문자열:  0 | 합계:  1
acmpyNeedMtr              NaN:  1 | 빈문자열:  0 | 합계:  1
etcAcmpyInfo              NaN:  1 | 빈문자열:  0 | 합계:  1
indoor_possible           NaN: 62 | 빈문자열:  0 | 합계: 62
outdoor_possible          NaN: 62 | 빈문자열:  0 | 합계: 62
homepage                  NaN:100 | 빈문자열:  0 | 합계:100
opertime                  NaN: 69 | 빈문자열:  0 | 합계: 69

결측 없는 컬럼: 6개


In [ ]:
print("=" * 55)
print("📊 주요 컬럼 분포")
print("=" * 55)

print("\n[data_source - 데이터 출처]")
print(raw_df["data_source"].value_counts())

print("\n[content_type_name - 장소 유형]")
print(raw_df["content_type_name"].value_counts())

print("\n[province - 시도별]")
print(raw_df["province"].value_counts())

print("\n[gu_name - 자치구/시군구 상위 20]")
print(raw_df["gu_name"].value_counts().head(20))

print("\n[acmpyTypeCd - 반려동물 동반유형]")
print(raw_df["acmpyTypeCd"].value_counts(dropna=False).head(10))

print("\n[acmpyPsblCpam - 동반가능동물]")
print(raw_df["acmpyPsblCpam"].value_counts(dropna=False).head(10))

In [10]:
print("=" * 55)
print("📍 좌표 데이터 확인")
print("=" * 55)

# 숫자형 변환 후 확인
mapx = pd.to_numeric(raw_df["mapx"], errors="coerce")
mapy = pd.to_numeric(raw_df["mapy"], errors="coerce")

print(f"mapx 결측: {mapx.isna().sum()}건 / {len(raw_df)}건")
print(f"mapy 결측: {mapy.isna().sum()}건 / {len(raw_df)}건")
print(f"\nmapx 범위: {mapx.min():.4f} ~ {mapx.max():.4f}")
print(f"mapy 범위: {mapy.min():.4f} ~ {mapy.max():.4f}")
print("\n📌 EDA 완료 → 전처리 방향:")
print("  1. acmpyPsblCpam → 견종 크기 표준화 (dog_size_category)")
print("  2. NaN + 빈 문자열 → '정보없음' 처리")
print("  3. acmpyTypeCd 출처별 표현 통일")
print("  4. has_pet_detail 컬럼 생성")
print("  5. 좌표 숫자형 변환")

📍 좌표 데이터 확인
mapx 결측: 0건 / 461건
mapy 결측: 0건 / 461건

mapx 범위: 126.8016 ~ 127.1573
mapy 범위: 37.4571 ~ 37.6622

📌 EDA 완료 → 전처리 방향:
  1. acmpyPsblCpam → 견종 크기 표준화 (dog_size_category)
  2. NaN + 빈 문자열 → '정보없음' 처리
  3. acmpyTypeCd 출처별 표현 통일
  4. has_pet_detail 컬럼 생성
  5. 좌표 숫자형 변환


## 📊 EDA 결과 요약

### 데이터 기본 현황
| 항목 | 내용 |
|---|---|
| 총 장소 수 | (재실행 후 확인) |
| 컬럼 수 | 17개 (province 포함) |
| 데이터 출처 | TourAPI + 문화시설CSV |
| 좌표 결측 | 0건 (완전함) |
| 전국 좌표 범위 | 경도 124.62 ~ 130.91 / 위도 33.12 ~ 38.58 |

---

### 결측값 현황
| 컬럼 | 결측 수 | 원인 |
|---|---|---|
| indoor_possible / outdoor_possible | TourAPI 건수 | TourAPI는 해당 컬럼 없음 |
| homepage | 다수 | 홈페이지 미등록 장소 |
| opertime | 다수 | 운영시간 미등록 |
| addr1 | 소수 | 주소 미등록 → 지도 표시 불가 |
| tel | 소수 | 전화번호 미등록 |
| acmpyTypeCd 외 4개 | 소수 | 상세정보 없음 |

---

### 장소 유형 분포
> 노트북 재실행 후 실제 수치로 업데이트

---

### 시도별 분포
> 전국 17개 시도 커버 (TourAPI 전국 수집 + 문화시설CSV 전국)  
> 노트북 재실행 후 시도별 상세 분포 확인

---

### 반려동물 동반유형 (acmpyTypeCd)
> ⚠️ 데이터 출처별 표현 방식 달라 전처리에서 통일 필요

| 값 | 출처 |
|---|---|
| 실내 동반가능 / 실외만 가능 | 문화시설CSV |
| 일부구역 동반가능 / 전구역 동반가능 | TourAPI |

## 4. 전처리
> EDA에서 파악한 문제점을 처리합니다.


In [ ]:
df = raw_df.copy()

# 4-1. 좌표 숫자형 변환 (필터 전에 먼저 실행)
# TourAPI는 JSON에서 문자열로 반환되므로 CSV(float)와 혼합 타입이 됨
df["mapx"] = pd.to_numeric(df["mapx"], errors="coerce")
df["mapy"] = pd.to_numeric(df["mapy"], errors="coerce")
print(f"✅ 좌표 변환 완료 (결측: {df[['mapx','mapy']].isna().any(axis=1).sum()}건)")

# 4-2. 한국 영토 범위 외 이상 좌표 제거
# 범위 근거: 실제 데이터 검증 결과 (위도 33.12~38.58, 경도 124.62~130.91)
KR_LAT_MIN, KR_LAT_MAX = 33.0, 39.0
KR_LON_MIN, KR_LON_MAX = 124.0, 132.0
before_coord = len(df)
df = df[
    df["mapy"].between(KR_LAT_MIN, KR_LAT_MAX) &
    df["mapx"].between(KR_LON_MIN, KR_LON_MAX)
]
removed = before_coord - len(df)
print(f"✅ 이상 좌표 제거: {removed}건 제거 (한국 범위 외) → {len(df)}건 남음")

# 4-3. 펫 상세정보 보유 여부 (NaN + 빈 문자열 모두 고려)
df["has_pet_detail"] = (
    df["acmpyTypeCd"].notna() & (df["acmpyTypeCd"] != "")
)
print(f"✅ 펫 상세정보 실제 보유 장소: {df['has_pet_detail'].sum()}건 / {len(df)}건")

# 4-4. 반려동물 관련 결측값 처리 (NaN + 빈 문자열 → '정보없음')
pet_cols = ["acmpyTypeCd", "acmpyPsblCpam", "acmpyNeedMtr", "etcAcmpyInfo"]
for col in pet_cols:
    if col in df.columns:
        df[col] = df[col].replace("", "정보없음").fillna("정보없음")
print("✅ 결측값 처리 완료")

# 4-5. 견종 크기 표준화
def classify_dog_size(val):
    val = str(val)
    if "불가" in val:                          return "입장불가"
    if "안내견" in val:                        return "안내견전용"
    if "전 견종" in val or "전견종" in val:    return "전견종"
    if "모두 가능" in val:                     return "전견종"
    if "대형" in val and "제외" in val:        return "소형/중형견"
    if "소형/중형" in val:                     return "소형/중형견"
    if "14kg" in val or "15kg" in val:         return "중형견이하"
    if "9kg" in val or "10kg" in val:          return "소형견"
    if "소형" in val or "훈련된" in val:       return "소형견"
    return "정보없음"

df["dog_size_category"] = df["acmpyPsblCpam"].apply(classify_dog_size)
print("✅ 견종 크기 표준화 완료")
print(df["dog_size_category"].value_counts())

# 4-6. 중복 최종 확인
before = len(df)
df = df.drop_duplicates(subset=["title", "gu_name"])
print(f"✅ 중복 제거: {before - len(df)}건 제거 → {len(df)}건 남음")

print(f"\n{'='*50}")
print(f"[전처리 완료] {len(df)}행 × {len(df.columns)}열")
print(f"{'='*50}")

In [12]:
# 4-6. indoor_possible / outdoor_possible 보완
# 방법 1: acmpyTypeCd로 역추론
def infer_indoor(row):
    if pd.notna(row["indoor_possible"]):  # 이미 값 있으면 유지
        return row["indoor_possible"]
    val = str(row["acmpyTypeCd"])
    if "전구역" in val:    return True
    if "실외만" in val:    return False
    if "실내" in val:      return True
    return None

def infer_outdoor(row):
    if pd.notna(row["outdoor_possible"]):  # 이미 값 있으면 유지
        return row["outdoor_possible"]
    val = str(row["acmpyTypeCd"])
    if "전구역" in val:    return True
    if "실외만" in val:    return True
    if "실내" in val:      return False
    return None

# 방법 2: etcAcmpyInfo 텍스트 키워드 파싱으로 추가 보완
def refine_indoor(row):
    if pd.notna(row["indoor_possible"]):
        return row["indoor_possible"]
    text = str(row["etcAcmpyInfo"]) + str(row["acmpyNeedMtr"])
    if any(k in text for k in ["실내", "내부", "매장 내"]):    return True
    if any(k in text for k in ["야외", "테라스", "외부만"]):   return False
    return None

def refine_outdoor(row):
    if pd.notna(row["outdoor_possible"]):
        return row["outdoor_possible"]
    text = str(row["etcAcmpyInfo"]) + str(row["acmpyNeedMtr"])
    if any(k in text for k in ["야외", "테라스", "외부"]):     return True
    if any(k in text for k in ["실내만", "내부만"]):           return False
    return None

# 적용 (1단계: acmpyTypeCd → 2단계: 텍스트 파싱)
df["indoor_possible"]  = df.apply(infer_indoor, axis=1)
df["indoor_possible"]  = df.apply(refine_indoor, axis=1)
df["outdoor_possible"] = df.apply(infer_outdoor, axis=1)
df["outdoor_possible"] = df.apply(refine_outdoor, axis=1)

# 보완 결과 확인
indoor_filled  = df["indoor_possible"].notna().sum()
outdoor_filled = df["outdoor_possible"].notna().sum()
print(f"✅ indoor_possible  보완 완료: {indoor_filled}건 / {len(df)}건 ({indoor_filled/len(df)*100:.1f}%)")
print(f"✅ outdoor_possible 보완 완료: {outdoor_filled}건 / {len(df)}건 ({outdoor_filled/len(df)*100:.1f}%)")
print(f"\n[indoor_possible 분포]")
print(df["indoor_possible"].value_counts(dropna=False))
print(f"\n[outdoor_possible 분포]")
print(df["outdoor_possible"].value_counts(dropna=False))

✅ indoor_possible  보완 완료: 427건 / 461건 (92.6%)
✅ outdoor_possible 보완 완료: 421건 / 461건 (91.3%)

[indoor_possible 분포]
indoor_possible
True     384
False     43
None      34
Name: count, dtype: int64

[outdoor_possible 분포]
outdoor_possible
False    287
True     134
None      40
Name: count, dtype: int64


## 📋 전처리 결과 요약

### 처리 내용
| 단계 | 처리 내용 | 결과 |
|---|---|---|
| 이상 좌표 제거 | 한국 영토 범위(위도 33~39, 경도 124~132) 외 제거 | 재실행 후 확인 |
| 펫 상세정보 여부 | NaN + 빈 문자열 동시 체크 | 재실행 후 확인 |
| 결측값 처리 | NaN + 빈 문자열 → '정보없음' | 4개 컬럼 처리 완료 |
| 견종 크기 표준화 | 비정형 텍스트 → 7개 분류 | dog_size_category 컬럼 생성 |
| 좌표 변환 | 문자열 → float | 재실행 후 확인 |
| 중복 제거 | 장소명 + 구명 기준 | 재실행 후 확인 |
| indoor_possible 보완 | acmpyTypeCd + 텍스트 파싱 | 재실행 후 확인 |
| outdoor_possible 보완 | acmpyTypeCd + 텍스트 파싱 | 재실행 후 확인 |

---

### 최종 데이터 현황
- **데이터 크기**: 재실행 후 확인 (전국 기준)
- **컬럼 수**: 18열 (province, dog_size_category, has_pet_detail 포함)

### 견종 크기 분류 기준
| 분류 | 조건 |
|---|---|
| 전견종 | "전 견종", "모두 가능" |
| 소형견 | "소형", "9~10kg" 이하 |
| 소형/중형견 | "소형/중형" |
| 중형견이하 | "14~15kg" 이하 |
| 안내견전용 | "안내견" |
| 입장불가 | "불가" |
| 정보없음 | 그 외 (향후 사장님 직접 등록으로 보완 예정) |

### 실내/실외 동반 추론 방식
1. **1단계**: `acmpyTypeCd` 텍스트 기반 (전구역/실내/실외만)
2. **2단계**: `etcAcmpyInfo` + `acmpyNeedMtr` 키워드 파싱 (실내·야외·테라스 등)

> 💡 두 단계 보완으로 90% 이상 coverage 확보 목표

## 5. 분석 및 시각화

In [13]:
print("=" * 55)
print("📊 최종 통계")
print("=" * 55)
print(f"총 장소 수            : {len(df)}곳")
print(f"펫 상세정보 있는 장소 : {df['has_pet_detail'].sum()}곳 ({df['has_pet_detail'].mean()*100:.1f}%)")
print(f"\n[데이터 출처별]")
print(df["data_source"].value_counts())
print(f"\n[카테고리별 장소 수]")
print(df.groupby("content_type_name").size().sort_values(ascending=False).to_string())
print(f"\n[자치구별 장소 수]")
print(df.groupby("gu_name").size().sort_values(ascending=False).to_string())
print(f"\n[견종 크기 분류]")
print(df["dog_size_category"].value_counts().to_string())


📊 최종 통계
총 장소 수            : 461곳
펫 상세정보 있는 장소 : 454곳 (98.5%)

[데이터 출처별]
data_source
문화시설CSV    399
TourAPI     62
Name: count, dtype: int64

[카테고리별 장소 수]
content_type_name
문화시설    195
음식점     166
관광지      77
숙박       18
쇼핑        3
레포츠       2

[자치구별 장소 수]
gu_name
종로구     67
강남구     49
마포구     39
송파구     32
용산구     29
중구      27
서초구     23
강서구     20
서대문구    16
성동구     16
강북구     14
관악구     14
노원구     13
성북구     11
강동구     11
광진구     10
양천구     10
은평구      9
동작구      9
도봉구      9
중랑구      8
동대문구     7
구로구      7
영등포구     7
금천구      4

[견종 크기 분류]
dog_size_category
전견종       226
정보없음      202
소형견        20
소형/중형견      6
안내견전용       3
중형견이하       2
입장불가        2


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("댕댕맵 - 전국 펫프렌들리 관광 데이터 분석", fontsize=16, fontweight="bold", fontproperties=prop)

# 차트 1: 시도별 장소 수
prov_stats = df.groupby("province").size().sort_values(ascending=False)
bars = axes[0,0].bar(prov_stats.index, prov_stats.values, color="#FF8C69")
axes[0,0].set_title("시도별 펫프렌들리 장소 수", fontproperties=prop)
axes[0,0].set_xlabel("시도", fontproperties=prop)
axes[0,0].set_ylabel("장소 수", fontproperties=prop)
axes[0,0].tick_params(axis="x", rotation=45)
for label in axes[0,0].get_xticklabels():
    label.set_fontproperties(prop)
for bar, val in zip(bars, prov_stats.values):
    axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                   str(val), ha="center", va="bottom", fontsize=8)

# 차트 2: 카테고리별 분포
type_stats = df.groupby("content_type_name").size()
colors = ["#FFB347","#87CEEB","#90EE90","#DDA0DD","#F08080","#98FB98"]
axes[0,1].pie(type_stats.values, labels=type_stats.index, autopct="%1.1f%%",
              colors=colors[:len(type_stats)], startangle=90,
              textprops={"fontproperties": prop})
axes[0,1].set_title("카테고리별 분포", fontproperties=prop)

# 차트 3: 견종 크기 분류
dog_stats = df["dog_size_category"].value_counts()
bars3 = axes[1,0].bar(dog_stats.index, dog_stats.values,
                      color=["#4ECDC4","#FF6B6B","#FFD700","#98FB98","#DDA0DD","#F08080","#87CEEB"])
axes[1,0].set_title("견종 크기별 입장 가능 장소", fontproperties=prop)
axes[1,0].set_xlabel("견종 분류", fontproperties=prop)
axes[1,0].set_ylabel("장소 수", fontproperties=prop)
axes[1,0].tick_params(axis="x", rotation=30)
for label in axes[1,0].get_xticklabels():
    label.set_fontproperties(prop)
for bar, val in zip(bars3, dog_stats.values):
    axes[1,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                   str(val), ha="center", va="bottom", fontsize=9)

# 차트 4: 데이터 출처별
source_stats = df["data_source"].value_counts()
axes[1,1].pie(source_stats.values, labels=source_stats.index, autopct="%1.1f%%",
              colors=["#87CEEB","#FFB347"], startangle=90,
              textprops={"fontproperties": prop})
axes[1,1].set_title("데이터 출처별 비율", fontproperties=prop)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/analysis_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ 차트 저장 완료")

## 📊 분석 및 시각화 결과 요약

### 전체 현황
| 항목 | 수치 |
|---|---|
| 총 장소 수 | (재실행 후 확인) |
| 펫 상세정보 보유 | (재실행 후 확인) |
| 데이터 출처 | TourAPI + 문화시설CSV |

---

### 차트 1. 시도별 펫프렌들리 장소 수
- 전국 17개 시도 커버
- 시도별 분포는 TourAPI 수집 건수와 문화시설CSV 비중에 따라 상이
> 💡 특정 시도 집중 현상 → AI 코스 추천 시 같은 시도 내 동선 구성

---

### 차트 2. 카테고리별 분포
| 유형 | 특성 |
|---|---|
| 문화시설 | 박물관·미술관 등 실내 중심 |
| 음식점 | 카페·레스토랑 |
| 관광지 | 공원·명소 등 야외 중심 |
| 숙박 | 반려동물 동반 가능 숙소 |
| 쇼핑·레포츠 | 소수 |

> 💡 문화시설·음식점 중심 → 반려견과 함께하는 **도심형 당일 코스** 설계에 최적화

---

### 차트 3. 견종 크기별 입장 가능 장소
| 분류 | 서비스 의미 |
|---|---|
| 전견종 | 모든 강아지 입장 가능 |
| 정보없음 | 사장님 등록으로 추후 보완 예정 |
| 소형견 | 소형견만 입장 가능 |
| 소형/중형견 | 소형·중형견 입장 가능 |
| 안내견전용 | 시각장애인 안내견만 가능 |
| 중형견이하 | 중형견 이하 입장 가능 |
| 입장불가 | 반려동물 입장 불가 |

> 💡 **정보없음** 비율이 높아 향후 사장님 직접 등록 시스템으로 보완 예정

---

### 차트 4. 데이터 출처별 비율
- 문화시설CSV (공식 반려동물 동반 시설 데이터) + TourAPI (반려동물 상세정보 보유)
> 💡 두 데이터 결합으로 단일 소스 대비 더 넓은 전국 커버리지 확보

## 6. 데이터 저장

In [ ]:
# 전처리 완료 데이터 저장
csv_path = f"{OUTPUT_DIR}/pet_tour_preprocessed.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"✅ 전처리 데이터 저장 : {csv_path}")

# 자치구별 통계 저장
area_stats = df.groupby("gu_name").size().sort_values(ascending=False)
stats_path = f"{OUTPUT_DIR}/area_stats.csv"
area_stats.reset_index().rename(columns={0: "count"}).to_csv(stats_path, index=False, encoding="utf-8-sig")
print(f"✅ 자치구별 통계 저장 : {stats_path}")

print(f"\n🐾 총 {len(df)}건의 전국 펫프렌들리 관광지 데이터 저장 완료!")
print(f"   - TourAPI     : {(df['data_source']=='TourAPI').sum()}건")
print(f"   - 문화시설CSV : {(df['data_source']=='문화시설CSV').sum()}건")

## 7. 서비스 기능 시연
> `app.py`에서 구현된 지도·필터·코스 추천 기능을 노트북 환경에서 시연합니다.  
> **AI 코스 추천**은 엔노이아 API 발급 후 연동 예정입니다.

| 기능 | 상태 |
|---|---|
| 견종 크기 호환성 필터 (`SIZE_COMPAT`) | ✅ 구현 완료 |
| 카테고리 / 시도 / 자치구 멀티 필터 | ✅ 구현 완료 |
| 카카오맵 지도 시각화 (전국, 마커 + 인포윈도우) | ✅ 구현 완료 |
| 규칙 기반 코스 추천 (시도별 동선) | ✅ 구현 완료 |
| AI 코스 추천 (엔노이아 API) | 🔜 API 발급 후 연동 예정 |

> **실행 방법**: 섹션 1~6을 순서대로 실행한 뒤 아래 셀 2개를 실행하면 인터랙티브 UI가 표시됩니다.

In [ ]:
import json
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

KAKAO_JS_KEY    = os.environ.get("KAKAO_JS_KEY", "")
ENNOEIA_KEY     = os.environ.get("ENNOEIA_API_KEY", "")
ENNOEIA_PROJECT = "KNTO-PROMPTON-2026-031"
ENNOEIA_HASH    = "be0cf3a4a803e6c2ecea872d15efecc4e9cc0231dc20f71bce65f843a4ad74ae"

SIZE_COMPAT = {
    "전체 (필터 없음)":   ["전견종", "소형견", "소형/중형견", "중형견이하", "안내견전용", "정보없음"],
    "소형견 (10kg 미만)": ["전견종", "소형견", "소형/중형견", "중형견이하", "정보없음"],
    "중형견 (25kg 미만)": ["전견종", "소형/중형견", "중형견이하", "정보없음"],
    "대형견":            ["전견종", "정보없음"],
}
COURSE_ORDER = ["관광지", "음식점", "문화시설", "레포츠", "숙박", "쇼핑", "기타"]
TIME_LABELS  = ["오전", "점심", "오후", "저녁"]

# 지도 중심 좌표: 실제 데이터 좌표 범위 중간값 기준
# 위도 33.12~38.58 → 중간 35.85 / 경도 124.62~130.91 → 중간 127.77
MAP_CENTER_LAT = 35.9
MAP_CENTER_LON = 127.8


def build_map_html(places, key):
    places_js = json.dumps(places, ensure_ascii=False)
    return (
        "<!DOCTYPE html><html><head>"
        "<meta charset='utf-8'>"
        "<style>"
        "body{margin:0;padding:0;font-family:'Malgun Gothic',sans-serif}"
        "#map{width:100%;height:520px}"
        ".iw{padding:10px;min-width:180px;max-width:230px}"
        ".iw-title{font-size:14px;font-weight:700;color:#111;margin-bottom:5px}"
        ".iw-addr{font-size:11px;color:#666;margin-bottom:6px;line-height:1.5}"
        ".tag{display:inline-block;padding:2px 8px;border-radius:10px;font-size:11px;margin-right:3px;color:#fff}"
        ".iw-etc{font-size:11px;color:#555;margin-top:5px;line-height:1.4}"
        "</style></head><body>"
        "<div id='map'></div>"
        "<script src='//dapi.kakao.com/v2/maps/sdk.js?appkey=" + key + "'></script>"
        "<script>"
        "var map=new kakao.maps.Map(document.getElementById('map'),{center:new kakao.maps.LatLng("
        + str(MAP_CENTER_LAT) + "," + str(MAP_CENTER_LON) + "),level:13});"
        "var iw=new kakao.maps.InfoWindow({zIndex:1});"
        "var C={'관광지':'#3B82F6','문화시설':'#8B5CF6','음식점':'#F97316','숙박':'#10B981','레포츠':'#EF4444','쇼핑':'#EC4899','기타':'#6B7280'};"
        "var S={'전견종':'#059669','소형견':'#0284C7','소형/중형견':'#7C3AED','중형견이하':'#B45309','정보없음':'#9CA3AF','안내견전용':'#6B7280','입장불가':'#DC2626'};"
        "var places=" + places_js + ";"
        "places.forEach(function(p){"
        "if(!p.mapy||!p.mapx)return;"
        "var cc=C[p.content_type_name]||'#6B7280';"
        "var sc=S[p.dog_size_category]||'#9CA3AF';"
        "var m=new kakao.maps.Marker({map:map,position:new kakao.maps.LatLng(p.mapy,p.mapx),title:p.title});"
        "var html='<div class=\"iw\">'"
        "+'<div class=\"iw-title\">'+p.title+'</div>'"
        "+'<div class=\"iw-addr\">'+(p.addr1||'')+'</div>'"
        "+'<div><span class=\"tag\" style=\"background:'+cc+'\">'+p.content_type_name+'</span>'"
        "+'<span class=\"tag\" style=\"background:'+sc+'\">'+p.dog_size_category+'</span></div>'"
        "+(p.acmpyTypeCd&&p.acmpyTypeCd!=='정보없음'?'<div class=\"iw-etc\">'+p.acmpyTypeCd+'</div>':'')"
        "+(p.tel&&p.tel!=='정보없음'?'<div class=\"iw-etc\">TEL: '+p.tel+'</div>':'')"
        "+'</div>';"
        "kakao.maps.event.addListener(m,'click',(function(mk,c){return function(){iw.setContent(c);iw.open(map,mk);};})(m,html));"
        "});"
        "</script></body></html>"
    )


def filter_df(dog_size_label, sel_cats, sel_provs, sel_gus):
    filtered = df.copy()
    filtered = filtered[filtered["dog_size_category"].isin(SIZE_COMPAT[dog_size_label])]
    if sel_cats:
        filtered = filtered[filtered["content_type_name"].isin(sel_cats)]
    if sel_provs:
        filtered = filtered[filtered["province"].isin(sel_provs)]
    if sel_gus:
        filtered = filtered[filtered["gu_name"].isin(sel_gus)]
    return filtered


def recommend_course(filtered_df):
    if len(filtered_df) == 0:
        return []
    valid = filtered_df[filtered_df["province"] != "정보없음"]
    top_prov = valid["province"].value_counts().index[0] if len(valid) > 0 else filtered_df["province"].value_counts().index[0]
    result = []
    for cat in COURSE_ORDER:
        cat_df = filtered_df[filtered_df["content_type_name"] == cat]
        if cat_df.empty:
            continue
        same_prov = cat_df[cat_df["province"] == top_prov]
        pool  = same_prov if len(same_prov) > 0 else cat_df
        place = pool.sample(1).iloc[0]
        addr  = place["addr1"] if pd.notna(place["addr1"]) else "주소 없음"
        result.append({
            "cat":   cat,
            "title": place["title"],
            "addr":  addr,
            "prov":  place["province"],
            "size":  place["dog_size_category"],
            "info":  place["acmpyTypeCd"],
        })
        if len(result) >= 4:
            break
    return result


def ai_recommend_course(dog_name, dog_size, filtered_df):
    if not ENNOEIA_KEY:
        return "ENNOEIA_API_KEY 미설정 -- .env에 추가 후 커널 재시작"

    sample = filtered_df.head(20)
    places_text = "\n".join(
        "- " + r["title"]
        + " | 유형: " + r["content_type_name"]
        + " | 시도: " + str(r["province"])
        + " | 자치구: " + r["gu_name"]
        + " | 주소: " + (r["addr1"] if pd.notna(r["addr1"]) else "정보없음")
        + " | 동반형태: " + r["acmpyTypeCd"]
        + " | 입장가능견종: " + r["dog_size_category"]
        for _, r in sample.iterrows()
    )
    prompt = (
        "아래 장소 목록은 한국관광공사 TourAPI와 공식 문화시설 데이터에서 검증된 반려동물 동반 가능 장소입니다.\n"
        "코스 추천 시 반드시 이 목록에 있는 장소를 우선적으로 활용하고,\n"
        "목록에 없는 장소를 추천할 경우 반드시 실제 존재하는 곳인지 확인 후 정확한 정보만 제공해주세요.\n\n"
        "강아지 정보: " + dog_name + " (" + dog_size + ")\n\n"
        "[검증된 반려동물 동반 가능 장소 목록]\n" + places_text + "\n\n"
        "위 목록을 참고해 오전·점심·오후 3~4곳의 하루 코스를 짜주세요.\n"
        "각 장소마다 (1) 장소명 (2) 주소 (3) 동반 조건 (4) 선정 이유(1줄)를 포함하고,\n"
        "이동 순서와 동선이 자연스럽도록 구성해 한국어로 답변해주세요."
    )
    resp = requests.post(
        "https://api.ennoia.so/api/preset/v2/chat/completions",
        headers={
            "project": ENNOEIA_PROJECT,
            "apiKey": ENNOEIA_KEY,
            "Content-Type": "application/json; charset=utf-8",
        },
        json={
            "hash": ENNOEIA_HASH,
            "params": {},
            "messages": [{"role": "user", "content": [{"type": "text", "text": prompt}]}],
        },
        timeout=90,
    )
    resp.raise_for_status()
    result = resp.json()
    if "choices" in result:
        content = result["choices"][0]["message"]["content"]
        if isinstance(content, list):
            return content[0].get("text", str(content))
        return str(content)
    return str(result)


if not KAKAO_JS_KEY:
    print("KAKAO_JS_KEY 미설정 -- .env에 추가 후 커널 재시작")
else:
    print("KAKAO_JS_KEY 로드됨")
if not ENNOEIA_KEY:
    print("ENNOEIA_API_KEY 미설정 -- .env에 추가 후 커널 재시작")
else:
    print("ENNOEIA_API_KEY 로드됨")
print("서비스 함수 정의 완료")


In [ ]:
w_name = widgets.Text(
    value="댕댕이", description="강아지 이름",
    layout=widgets.Layout(width="300px")
)
w_size = widgets.RadioButtons(
    options=list(SIZE_COMPAT.keys()),
    description="견종 크기",
    layout=widgets.Layout(width="320px"),
)
w_cats = widgets.SelectMultiple(
    options=sorted(df["content_type_name"].unique()),
    description="카테고리",
    rows=6,
    layout=widgets.Layout(width="210px"),
)
w_provs = widgets.SelectMultiple(
    options=sorted(df["province"].dropna().unique()),
    description="시도",
    rows=6,
    layout=widgets.Layout(width="160px"),
)
w_gus = widgets.SelectMultiple(
    options=sorted(df["gu_name"].dropna().unique()),
    description="자치구",
    rows=6,
    layout=widgets.Layout(width="180px"),
)

def on_provs_change(change):
    selected_provs = list(w_provs.value)
    if selected_provs:
        filtered_gus = sorted(df[df["province"].isin(selected_provs)]["gu_name"].dropna().unique())
    else:
        filtered_gus = sorted(df["gu_name"].dropna().unique())
    # value를 먼저 초기화한 뒤 options 교체 (순서가 바뀌면 validation 오류 발생)
    w_gus.value   = ()
    w_gus.options = filtered_gus

w_provs.observe(on_provs_change, names="value")

w_map_btn    = widgets.Button(description="지도 보기",    button_style="info",    layout=widgets.Layout(width="150px"))
w_course_btn = widgets.Button(description="코스 추천",   button_style="success", layout=widgets.Layout(width="150px"))
w_ai_btn     = widgets.Button(description="AI 코스 추천", button_style="warning", layout=widgets.Layout(width="150px"))
w_out        = widgets.Output()


def _get_filtered():
    return filter_df(w_size.value, list(w_cats.value), list(w_provs.value), list(w_gus.value))


def _show_stats(filtered):
    print("검색 결과: " + str(len(filtered)) + "곳 | "
          + str(filtered["province"].nunique()) + "개 시도 | "
          + str(filtered["content_type_name"].nunique()) + "개 카테고리")


def on_map_click(b):
    with w_out:
        clear_output(wait=True)
        filtered = _get_filtered()
        _show_stats(filtered)
        if not KAKAO_JS_KEY:
            print("KAKAO_JS_KEY 미설정 -- .env에 추가 후 커널 재시작")
            return
        if len(filtered) == 0:
            print("조건에 맞는 장소가 없습니다.")
            return
        cols = ["title", "addr1", "mapx", "mapy", "content_type_name",
                "province", "gu_name", "dog_size_category", "acmpyTypeCd", "tel"]
        places_df = filtered[cols].copy()
        places_df["tel"] = places_df["tel"].replace("nan", "정보없음").fillna("정보없음")
        places = places_df.to_dict("records")
        display(HTML(build_map_html(places, KAKAO_JS_KEY)))


def on_course_click(b):
    with w_out:
        clear_output(wait=True)
        filtered = _get_filtered()
        _show_stats(filtered)
        name   = w_name.value or "댕댕이"
        course = recommend_course(filtered)
        if not course:
            print("조건에 맞는 장소가 없어 코스를 만들 수 없습니다.")
            return
        print(name + "를 위한 추천 코스")
        print("-" * 40)
        for i, p in enumerate(course):
            label = TIME_LABELS[i] if i < len(TIME_LABELS) else str(i + 1)
            print(label + "  " + p["title"] + "  [" + p["cat"] + "]")
            print("    " + p["addr"] + " / " + p["size"])
            if p["info"] != "정보없음":
                print("    " + p["info"])
            print()


def on_ai_click(b):
    with w_out:
        clear_output(wait=True)
        filtered = _get_filtered()
        _show_stats(filtered)
        name   = w_name.value or "댕댕이"
        result = ai_recommend_course(name, w_size.value, filtered)
        print(result)


w_map_btn.on_click(on_map_click)
w_course_btn.on_click(on_course_click)
w_ai_btn.on_click(on_ai_click)

profile_box = widgets.VBox([
    widgets.HTML("<b>강아지 프로필</b>"),
    w_name, w_size,
], layout=widgets.Layout(margin="0 30px 0 0"))

filter_box = widgets.VBox([
    widgets.HTML("<b>필터</b> <small>(Ctrl/Cmd + 클릭으로 복수 선택)</small>"),
    widgets.HBox([w_cats, w_provs, w_gus]),
])

btn_box = widgets.HBox(
    [w_map_btn, w_course_btn, w_ai_btn],
    layout=widgets.Layout(margin="12px 0"),
)

display(widgets.VBox([
    widgets.HBox([profile_box, filter_box]),
    btn_box,
    w_out,
]))

In [21]:

# 문화시설 CSV 로드 및 전처리 (전국)
csv_raw = pd.read_csv(
    "전국_반려동물_동반_가능_문화시설_위치_데이터_2023_.csv",
    encoding="utf-8"
)

PROV_MAP = {
    "서울특별시": "서울", "부산광역시": "부산", "대구광역시": "대구",
    "인천광역시": "인천", "광주광역시": "광주", "대전광역시": "대전",
    "울산광역시": "울산", "세종특별자치시": "세종", "경기도": "경기",
    "강원특별자치도": "강원", "강원도": "강원",
    "충청북도": "충북", "충청남도": "충남",
    "전북특별자치도": "전북", "전라북도": "전북",
    "전라남도": "전남", "경상북도": "경북",
    "경상남도": "경남", "제주특별자치도": "제주",
}

csv_all = csv_raw.copy()
print(f"전국 로드 완료: {len(csv_all)}건")
print(csv_all["CTPRVN_NM"].value_counts().head(10))

cat_map = {
    "카페": "음식점", "식당": "음식점", "레스토랑": "음식점",
    "박물관": "문화시설", "미술관": "문화시설", "문예회관": "문화시설",
    "도서관": "문화시설", "전시관": "문화시설",
    "여행지": "관광지", "공원": "관광지", "해수욕장": "관광지",
    "산": "관광지", "캠핑장": "관광지",
    "펜션": "숙박", "호텔": "숙박", "모텔": "숙박", "리조트": "숙박",
    "애견호텔": "숙박",
}

csv_df = pd.DataFrame({
    "title"           : csv_all["FCLTY_NM"].values,
    "addr1"           : csv_all["RDNMADR_NM"].values,
    "province"        : csv_all["CTPRVN_NM"].map(PROV_MAP).fillna(csv_all["CTPRVN_NM"]).values,
    "gu_name"         : csv_all["SIGNGU_NM"].values,
    "mapx"            : pd.to_numeric(csv_all["LC_LO"], errors="coerce").values,
    "mapy"            : pd.to_numeric(csv_all["LC_LA"], errors="coerce").values,
    "tel"             : csv_all["TEL_NO"].astype(str).values,
    "acmpyTypeCd"     : csv_all["IN_PLACE_ACP_POSBL_AT"].map({"Y": "실내 동반가능", "N": "실외만 가능"}).values,
    "acmpyPsblCpam"   : csv_all["ENTRN_POSBL_PET_SIZE_VALUE"].values,
    "acmpyNeedMtr"    : csv_all["PET_LMTT_MTR_CN"].values,
    "etcAcmpyInfo"    : csv_all["PET_INFO_CN"].values,
    "indoor_possible" : (csv_all["IN_PLACE_ACP_POSBL_AT"] == "Y").values,
    "outdoor_possible": (csv_all["OUT_PLACE_ACP_POSBL_AT"] == "Y").values,
    "homepage"        : csv_all["HMPG_URL"].values,
    "opertime"        : csv_all["OPER_TIME"].values,
    "content_type_name": csv_all["CTGRY_THREE_NM"].map(cat_map).fillna("기타").values,
    "data_source"     : "문화시설CSV",
})

print(f"\n✅ CSV 전처리 완료: {len(csv_df)}행 x {len(csv_df.columns)}열")
print("\n[시도별 현황 상위 10]")
print(csv_df["province"].value_counts().head(10))


전국 로드 완료: 23929건
CTPRVN_NM
경기도      6485
서울특별시    4367
경상남도     1462
부산광역시    1451
인천광역시    1347
경상북도     1125
충청남도      993
대구광역시     988
전라북도      902
전라남도      900
Name: count, dtype: int64

✅ CSV 전처리 완료: 23929행 x 17열

[시도별 현황 상위 10]
province
경기    6485
서울    4367
경남    1462
부산    1451
인천    1347
경북    1125
충남     993
대구     988
전북     902
전남     900
Name: count, dtype: int64


In [22]:

import re

tour_df["indoor_possible"]  = None
tour_df["outdoor_possible"] = None
tour_df["homepage"]         = None
tour_df["opertime"]         = None
tour_df["data_source"]      = "TourAPI"

common_cols = ["title", "addr1", "province", "gu_name", "mapx", "mapy", "tel",
               "acmpyTypeCd", "acmpyPsblCpam", "acmpyNeedMtr", "etcAcmpyInfo",
               "indoor_possible", "outdoor_possible",
               "homepage", "opertime", "content_type_name", "data_source"]

def province_from_addr(addr):
    if pd.isna(addr): return None
    for full, short in PROV_MAP.items():
        if full in str(addr): return short
    return None

def gu_from_addr(addr):
    if pd.isna(addr): return None
    s = str(addr)
    m = re.search(r'([가-힣]+[구군])\b', s)
    if m: return m.group(1)
    m = re.search(r'([가-힣]{2,5}시)\b', s)
    if m and not any(m.group(1) in k for k in PROV_MAP):
        return m.group(1)
    return None

tour_df["province"] = tour_df["addr1"].apply(province_from_addr).fillna(tour_df["area_name"])
tour_df["gu_name"]  = tour_df["addr1"].apply(gu_from_addr).fillna("정보없음")

type_map = {"12": "관광지", "14": "문화시설", "32": "숙박",
            "39": "음식점", "28": "레포츠",  "38": "쇼핑"}
tour_df["content_type_name"] = tour_df["contenttypeid"].astype(str).map(type_map).fillna("기타")

tour_standard = tour_df[common_cols].copy()
csv_standard  = csv_df[common_cols].copy()

raw_df = pd.concat([tour_standard, csv_standard], ignore_index=True)
before = len(raw_df)
raw_df = raw_df.drop_duplicates(subset=["title", "gu_name"], keep="first")

print(f"중복 제거: {before - len(raw_df)}건 제거")
print(f"\n✅ 통합 완료 (raw_df): {len(raw_df)}건")
print(f"  - TourAPI     : {(raw_df['data_source']=='TourAPI').sum()}건")
print(f"  - 문화시설CSV : {(raw_df['data_source']=='문화시설CSV').sum()}건")
print(f"\n[시도별 현황 상위 10]")
print(raw_df["province"].value_counts().head(10))
print(f"\n[카테고리별 현황]")
print(raw_df["content_type_name"].value_counts())


중복 제거: 155건 제거

✅ 통합 완료 (raw_df): 24486건
  - TourAPI     : 711건
  - 문화시설CSV : 23775건

[시도별 현황 상위 10]
province
경기    6542
서울    4419
경남    1486
부산    1450
인천    1368
경북    1172
강원    1051
충남    1016
대구     995
전남     952
Name: count, dtype: int64

[카테고리별 현황]
content_type_name
기타      20302
문화시설     1380
관광지      1235
음식점       878
숙박        605
레포츠        66
쇼핑         20
Name: count, dtype: int64


In [23]:

# EDA
print("=" * 55)
print("📋 데이터 기본 정보")
print("=" * 55)
print(f"데이터 크기  : {raw_df.shape[0]}행 × {raw_df.shape[1]}열")
print(f"\n컬럼 목록: {list(raw_df.columns)}")

print("\n" + "=" * 55)
print("🔍 결측값 현황")
print("=" * 55)
for col in raw_df.columns:
    null_cnt  = raw_df[col].isna().sum()
    empty_cnt = (raw_df[col] == "").sum() if raw_df[col].dtype == object else 0
    total     = null_cnt + empty_cnt
    if total > 0:
        print(f"{col:<25} NaN:{null_cnt:5} | 빈문자열:{empty_cnt:4} | 합계:{total:5}")

print("\n" + "=" * 55)
print("📍 좌표 데이터 확인")
print("=" * 55)
mapx = pd.to_numeric(raw_df["mapx"], errors="coerce")
mapy = pd.to_numeric(raw_df["mapy"], errors="coerce")
print(f"mapx 결측: {mapx.isna().sum()}건 / {len(raw_df)}건")
print(f"mapy 결측: {mapy.isna().sum()}건 / {len(raw_df)}건")
print(f"mapx 범위: {mapx.min():.4f} ~ {mapx.max():.4f}")
print(f"mapy 범위: {mapy.min():.4f} ~ {mapy.max():.4f}")


📋 데이터 기본 정보
데이터 크기  : 24486행 × 17열

컬럼 목록: ['title', 'addr1', 'province', 'gu_name', 'mapx', 'mapy', 'tel', 'acmpyTypeCd', 'acmpyPsblCpam', 'acmpyNeedMtr', 'etcAcmpyInfo', 'indoor_possible', 'outdoor_possible', 'homepage', 'opertime', 'content_type_name', 'data_source']

🔍 결측값 현황
addr1                     NaN:  295 | 빈문자열:   0 | 합계:  295
gu_name                   NaN:  145 | 빈문자열:   0 | 합계:  145
tel                       NaN: 1508 | 빈문자열:   0 | 합계: 1508
acmpyTypeCd               NaN:    6 | 빈문자열:   0 | 합계:    6
acmpyPsblCpam             NaN:    6 | 빈문자열:   0 | 합계:    6
acmpyNeedMtr              NaN:    6 | 빈문자열:   0 | 합계:    6
etcAcmpyInfo              NaN:    6 | 빈문자열:   0 | 합계:    6
indoor_possible           NaN:  711 | 빈문자열:   0 | 합계:  711
outdoor_possible          NaN:  711 | 빈문자열:   0 | 합계:  711
homepage                  NaN:13952 | 빈문자열:   0 | 합계:13952
opertime                  NaN: 2738 | 빈문자열:   0 | 합계: 2738

📍 좌표 데이터 확인
mapx 결측: 0건 / 24486건
mapy 결측: 0건 / 24486건
mapx 범위: 117.99

In [24]:

# 전처리
df = raw_df.copy()

# 4-1. 좌표 숫자형 변환
df["mapx"] = pd.to_numeric(df["mapx"], errors="coerce")
df["mapy"] = pd.to_numeric(df["mapy"], errors="coerce")
print(f"✅ 좌표 변환 완료 (결측: {df[['mapx','mapy']].isna().any(axis=1).sum()}건)")

# 4-2. 한국 영토 범위 외 이상 좌표 제거
before_coord = len(df)
df = df[df["mapy"].between(33.0, 39.0) & df["mapx"].between(124.0, 132.0)]
print(f"✅ 이상 좌표 제거: {before_coord - len(df)}건 제거 → {len(df)}건 남음")

# 4-3. 펫 상세정보 보유 여부
df["has_pet_detail"] = df["acmpyTypeCd"].notna() & (df["acmpyTypeCd"] != "")
print(f"✅ 펫 상세정보 실제 보유 장소: {df['has_pet_detail'].sum()}건 / {len(df)}건")

# 4-4. 결측값 처리
for col in ["acmpyTypeCd", "acmpyPsblCpam", "acmpyNeedMtr", "etcAcmpyInfo"]:
    if col in df.columns:
        df[col] = df[col].replace("", "정보없음").fillna("정보없음")
print("✅ 결측값 처리 완료")

# 4-5. 견종 크기 표준화
def classify_dog_size(val):
    val = str(val)
    if "불가" in val:                          return "입장불가"
    if "안내견" in val:                        return "안내견전용"
    if "전 견종" in val or "전견종" in val:    return "전견종"
    if "모두 가능" in val:                     return "전견종"
    if "대형" in val and "제외" in val:        return "소형/중형견"
    if "소형/중형" in val:                     return "소형/중형견"
    if "14kg" in val or "15kg" in val:         return "중형견이하"
    if "9kg" in val or "10kg" in val:          return "소형견"
    if "소형" in val or "훈련된" in val:       return "소형견"
    return "정보없음"

df["dog_size_category"] = df["acmpyPsblCpam"].apply(classify_dog_size)
print("✅ 견종 크기 표준화 완료")
print(df["dog_size_category"].value_counts())

# 4-6. 중복 최종 확인
before = len(df)
df = df.drop_duplicates(subset=["title", "gu_name"])
print(f"\n✅ 중복 제거: {before - len(df)}건 제거 → {len(df)}건 남음")
print(f"\n{'='*50}")
print(f"[전처리 완료] {len(df)}행 × {len(df.columns)}열")
print(f"{'='*50}")


✅ 좌표 변환 완료 (결측: 0건)
✅ 이상 좌표 제거: 2건 제거 → 24484건 남음
✅ 펫 상세정보 실제 보유 장소: 24438건 / 24484건
✅ 결측값 처리 완료
✅ 견종 크기 표준화 완료
dog_size_category
전견종       20475
정보없음       3274
소형견         460
소형/중형견      179
중형견이하        62
입장불가         22
안내견전용        12
Name: count, dtype: int64

✅ 중복 제거: 0건 제거 → 24484건 남음

[전처리 완료] 24484행 × 19열


In [25]:

# indoor/outdoor 보완
def infer_indoor(row):
    if pd.notna(row["indoor_possible"]): return row["indoor_possible"]
    val = str(row["acmpyTypeCd"])
    if "전구역" in val: return True
    if "실외만" in val: return False
    if "실내" in val:   return True
    return None

def infer_outdoor(row):
    if pd.notna(row["outdoor_possible"]): return row["outdoor_possible"]
    val = str(row["acmpyTypeCd"])
    if "전구역" in val: return True
    if "실외만" in val: return True
    if "실내" in val:   return False
    return None

def refine_indoor(row):
    if pd.notna(row["indoor_possible"]): return row["indoor_possible"]
    text = str(row["etcAcmpyInfo"]) + str(row["acmpyNeedMtr"])
    if any(k in text for k in ["실내", "내부", "매장 내"]): return True
    if any(k in text for k in ["야외", "테라스", "외부만"]): return False
    return None

def refine_outdoor(row):
    if pd.notna(row["outdoor_possible"]): return row["outdoor_possible"]
    text = str(row["etcAcmpyInfo"]) + str(row["acmpyNeedMtr"])
    if any(k in text for k in ["야외", "테라스", "외부"]): return True
    if any(k in text for k in ["실내만", "내부만"]): return False
    return None

df["indoor_possible"]  = df.apply(infer_indoor,   axis=1)
df["indoor_possible"]  = df.apply(refine_indoor,  axis=1)
df["outdoor_possible"] = df.apply(infer_outdoor,  axis=1)
df["outdoor_possible"] = df.apply(refine_outdoor, axis=1)

print(f"✅ indoor_possible  보완: {df['indoor_possible'].notna().sum()}건 / {len(df)}건")
print(f"✅ outdoor_possible 보완: {df['outdoor_possible'].notna().sum()}건 / {len(df)}건")


✅ indoor_possible  보완: 24150건 / 24484건
✅ outdoor_possible 보완: 24117건 / 24484건


In [26]:

# 최종 통계
print("=" * 55)
print("📊 최종 통계")
print("=" * 55)
print(f"총 장소 수            : {len(df)}곳")
print(f"펫 상세정보 있는 장소 : {df['has_pet_detail'].sum()}곳 ({df['has_pet_detail'].mean()*100:.1f}%)")
print(f"\n[데이터 출처별]")
print(df["data_source"].value_counts())
print(f"\n[시도별 장소 수 상위 10]")
print(df.groupby("province").size().sort_values(ascending=False).head(10).to_string())
print(f"\n[카테고리별 장소 수]")
print(df.groupby("content_type_name").size().sort_values(ascending=False).to_string())
print(f"\n[견종 크기 분류]")
print(df["dog_size_category"].value_counts().to_string())


📊 최종 통계
총 장소 수            : 24484곳
펫 상세정보 있는 장소 : 24438곳 (99.8%)

[데이터 출처별]
data_source
문화시설CSV    23775
TourAPI      709
Name: count, dtype: int64

[시도별 장소 수 상위 10]
province
경기    6540
서울    4419
경남    1486
부산    1450
인천    1368
경북    1172
강원    1051
충남    1016
대구     995
전남     952

[카테고리별 장소 수]
content_type_name
기타      20302
문화시설     1380
관광지      1233
음식점       878
숙박        605
레포츠        66
쇼핑         20

[견종 크기 분류]
dog_size_category
전견종       20475
정보없음       3274
소형견         460
소형/중형견      179
중형견이하        62
입장불가         22
안내견전용        12


In [27]:

# 데이터 저장
import os
OUTPUT_DIR = "./data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_path = f"{OUTPUT_DIR}/pet_tour_preprocessed.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"✅ 전처리 데이터 저장 : {csv_path}")

area_stats = df.groupby("gu_name").size().sort_values(ascending=False)
stats_path = f"{OUTPUT_DIR}/area_stats.csv"
area_stats.reset_index().rename(columns={0: "count"}).to_csv(stats_path, index=False, encoding="utf-8-sig")
print(f"✅ 자치구별 통계 저장 : {stats_path}")

print(f"\n🐾 총 {len(df)}건의 전국 펫프렌들리 관광지 데이터 저장 완료!")
print(f"   - TourAPI     : {(df['data_source']=='TourAPI').sum()}건")
print(f"   - 문화시설CSV : {(df['data_source']=='문화시설CSV').sum()}건")


✅ 전처리 데이터 저장 : ./data/pet_tour_preprocessed.csv
✅ 자치구별 통계 저장 : ./data/area_stats.csv

🐾 총 24484건의 전국 펫프렌들리 관광지 데이터 저장 완료!
   - TourAPI     : 709건
   - 문화시설CSV : 23775건


In [28]:

import pandas as pd
from pathlib import Path

csv_path = Path("./data/pet_tour_preprocessed.csv")
if csv_path.exists():
    check = pd.read_csv(csv_path)
    print(f"✅ 파일 존재 확인")
    print(f"   행 수      : {len(check)}")
    print(f"   컬럼 수    : {len(check.columns)}")
    print(f"   컬럼 목록  : {list(check.columns)}")
    print(f"\n[시도별 상위 5]")
    print(check["province"].value_counts().head(5).to_string())
    print(f"\n[카테고리별]")
    print(check["content_type_name"].value_counts().to_string())
else:
    print("❌ 파일이 존재하지 않습니다.")


✅ 파일 존재 확인
   행 수      : 24484
   컬럼 수    : 19
   컬럼 목록  : ['title', 'addr1', 'province', 'gu_name', 'mapx', 'mapy', 'tel', 'acmpyTypeCd', 'acmpyPsblCpam', 'acmpyNeedMtr', 'etcAcmpyInfo', 'indoor_possible', 'outdoor_possible', 'homepage', 'opertime', 'content_type_name', 'data_source', 'has_pet_detail', 'dog_size_category']

[시도별 상위 5]
province
경기    6540
서울    4419
경남    1486
부산    1450
인천    1368

[카테고리별]
content_type_name
기타      20302
문화시설     1380
관광지      1233
음식점       878
숙박        605
레포츠        66
쇼핑         20


In [29]:

import requests, os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(Path("../.env"))
key     = os.environ.get("ENNOEIA_API_KEY", "")
project = "KNTO-PROMPTON-2026-121"
hash_   = "da9313cd70180ac3cfc1b38f54973cc0d40628fee3a572e2eb6d9cf5a3e6a6dc"

print(f"키 앞 8자리  : {key[:8]}...")
print(f"키 길이      : {len(key)}")

# 실제 API 호출 테스트
resp = requests.post(
    "https://api.ennoia.so/api/preset/v2/chat/completions",
    headers={"project": project, "apiKey": key, "Content-Type": "application/json; charset=utf-8"},
    json={
        "hash": hash_,
        "params": {},
        "messages": [{"role": "user", "content": [{"type": "text", "text": "테스트"}]}]
    },
    timeout=30,
)
print(f"\nHTTP 상태코드 : {resp.status_code}")
print(f"응답 내용     : {resp.text[:300]}")


키 앞 8자리  : 2471a6f7...
키 길이      : 64

HTTP 상태코드 : 401
응답 내용     : {"timestamp":1780404192953,"error_code":40101,"error_type":"INVALID_API_KEY","message":"Invalid apiKey - KNTO-PROMPTON-2026-121, 2471a6f725b9faeeb95cd8d3b309f5c57916bcfcc5e4aabcd9dd0dd427b60d8d"}


In [30]:

import requests

# 새 키로 직접 테스트
key     = "d23b45468f4103828f5fc7980ccfc757addf9c879f032d7e153839daac9d3644"
project = "KNTO-PROMPTON-2026-121"
hash_   = "da9313cd70180ac3cfc1b38f54973cc0d40628fee3a572e2eb6d9cf5a3e6a6dc"

resp = requests.post(
    "https://api.ennoia.so/api/preset/v2/chat/completions",
    headers={"project": project, "apiKey": key, "Content-Type": "application/json; charset=utf-8"},
    json={
        "hash": hash_,
        "params": {},
        "messages": [{"role": "user", "content": [{"type": "text", "text": "안녕"}]}]
    },
    timeout=30,
)
print(f"HTTP 상태코드 : {resp.status_code}")
print(f"응답          : {resp.text[:300]}")


HTTP 상태코드 : 200
응답          : {"id":"chatcmpl-58f36d8ea7d64e8dac472844","object":"chat.completion.chunk","created":1780404210534,"model":"anthropic/claude-sonnet-4-5","choices":[{"index":0,"message":{"role":"assistant","content":[{"type":"text","text":"안녕하세요! \uD83D\uDC3E\n\n저는 서울 반려견 동반 여행 코스를 추천하는 **댕댕맵**이에요!\n\n강아지와 함께 즐거운 하루


## 추가 진행할 사항
- Streamlit Cloud 배포 / 
GitHub에 코드를 올리고 https://share.streamlit.io 에서 배포하면 누구나 링크로 접속 가능

In [31]:

# 꽃개애견카페 데이터 직접 확인
result = df[df["title"].str.contains("꽃개", na=False)]
print(result[["title","addr1","province","gu_name","mapx","mapy","data_source"]].to_string())


             title                     addr1 province  gu_name        mapx       mapy data_source
3336            꽃개        서울특별시 송파구 중대로33길 4       서울      송파구  127.131060  37.505773     문화시설CSV
3337          꽃개꽃냥  경기도 김포시 김포한강8로148번길 82-2       경기      김포시  126.645978  37.634271     문화시설CSV
3338   꽃개냥 반려동물용품점         인천광역시 부평구 부평대로 91       인천      부평구  126.721720  37.499554     문화시설CSV
3339       꽃개애견미용실     충청북도 청주시 서원구 사직대로 258       충북  청주시 서원구  127.476078  36.635351     문화시설CSV
3340       꽃개애견미용실           경상남도 양산시 양주로 32       경남      양산시  129.026852  35.330489     문화시설CSV
3341        꽃개애견카페          대구광역시 달서구 상인로 84       대구      달서구  128.547744  35.814732     문화시설CSV
3342         꽃개하우스            경기도 김포시 솔터로 22       경기      김포시  126.630858  37.645429     문화시설CSV
11547        살롱드꽃개         서울특별시 양천구 목동동로 10       서울      양천구  126.861310  37.507758     문화시설CSV
14702     애견미용실 꽃개    충청남도 아산시 배방읍 배방로 57-29       충남      아산시  127.057707  36.772554     문화시설CSV


In [32]:

# 원본 CSV에서 꽃개애견카페 확인
raw_entry = csv_raw[csv_raw["FCLTY_NM"].str.contains("꽃개애견카페", na=False)]
print(raw_entry[["FCLTY_NM","CTPRVN_NM","SIGNGU_NM","RDNMADR_NM","LC_LA","LC_LO"]].to_string())


     FCLTY_NM CTPRVN_NM SIGNGU_NM        RDNMADR_NM      LC_LA       LC_LO
2629   꽃개애견카페     대구광역시       달서구  대구광역시 달서구 상인로 84  35.814732  128.547744


In [33]:

# 전체 데이터에서 충북 음성 지역 확인
print("=== 충북 음성 지역 전체 데이터 ===")
eumseong = df[(df["province"] == "충북") & (df["gu_name"].str.contains("음성", na=False))]
print(f"음성 데이터: {len(eumseong)}건")
print(eumseong[["title","addr1","mapx","mapy"]].head(10).to_string())

print("\n=== 좌표 범위로 충북 음성 해당 좌표 검증 ===")
# 충북 음성군 좌표 범위: 위도 36.8~37.1, 경도 127.5~127.8 근방
music = df[(df["mapy"].between(36.8, 37.2)) & (df["mapx"].between(127.5, 127.9))]
print(f"해당 좌표 범위 장소: {len(music)}건")

print("\n=== 원본 CSV 시도별 건수 ===")
print(csv_raw["CTPRVN_NM"].value_counts().to_string())


=== 충북 음성 지역 전체 데이터 ===
음성 데이터: 46건
                title                     addr1        mapx       mapy
447        삼성 반려견 놀이터          충청북도 음성군 삼성면 양덕리  127.486618  37.024979
451        원남 반려견 놀이터          충청북도 음성군 원남면 조촌리  127.609620  36.871468
1182         감우재전승기념관     충청북도 음성군 음성읍 생음대로 594  127.647013  36.950683
1622           개스트하우스  충청북도 음성군 감곡면 장감로132번길 25  127.639876  37.118332
1767             건강약국      충청북도 음성군 금왕읍 무극로 278  127.592424  36.992615
2904        그린벨동물의료센터       충청북도 음성군 맹동면 대하1길 4  127.541060  36.907810
3020           금왕동물병원     충청북도 음성군 금왕읍 대금로 1498  127.598483  36.989210
3021           금왕종로약국      충청북도 음성군 금왕읍 무극로 284  127.593347  36.992742
3666  남영가축병원 출장진료전문병원   충청북도 음성군 음성읍 중앙로40번길 24  127.692015  36.927578
3740          낼름낼름펫푸드       충청북도 음성군 맹동면 장성1길 9  127.537149  36.910509

=== 좌표 범위로 충북 음성 해당 좌표 검증 ===
해당 좌표 범위 장소: 92건

=== 원본 CSV 시도별 건수 ===
CTPRVN_NM
경기도        6485
서울특별시      4367
경상남도       1462
부산광역시      1451
인천광역시      1347
경상북도       1125
충청남도  

In [34]:

# 시도별 좌표 유효 범위 정의
PROV_BBOX = {
    "서울": (37.41, 37.70, 126.73, 127.18),
    "인천": (37.25, 37.80, 126.35, 126.82),
    "경기": (36.90, 38.30, 126.35, 127.82),
    "강원": (36.98, 38.62, 127.05, 129.36),
    "충북": (36.35, 37.28, 127.36, 128.50),
    "충남": (35.90, 37.10, 125.90, 127.56),
    "대전": (36.19, 36.52, 127.25, 127.57),
    "세종": (36.38, 36.64, 127.18, 127.40),
    "경북": (35.57, 37.16, 128.02, 129.59),
    "경남": (34.66, 35.67, 127.59, 129.46),
    "대구": (35.66, 36.10, 128.31, 128.80),
    "부산": (35.00, 35.40, 128.74, 129.32),
    "울산": (35.35, 35.72, 128.98, 129.49),
    "전북": (35.37, 36.00, 126.34, 127.83),
    "전남": (33.95, 35.44, 125.87, 127.74),
    "광주": (35.05, 35.26, 126.72, 127.02),
    "제주": (33.10, 33.65, 126.10, 126.98),
}

def is_coord_valid(row):
    prov = row["province"]
    if prov not in PROV_BBOX: return True  # 범위 없으면 통과
    lat_min, lat_max, lon_min, lon_max = PROV_BBOX[prov]
    return (lat_min <= row["mapy"] <= lat_max) and (lon_min <= row["mapx"] <= lon_max)

before = len(df)
valid_mask = df.apply(is_coord_valid, axis=1)
invalid_df = df[~valid_mask]
df = df[valid_mask].reset_index(drop=True)

print(f"주소↔좌표 불일치 제거: {before - len(df)}건")
print(f"최종 데이터: {len(df)}건\n")
print("=== 제거된 불일치 데이터 샘플 (상위 10건) ===")
print(invalid_df[["title","province","gu_name","addr1","mapx","mapy"]].head(10).to_string())


주소↔좌표 불일치 제거: 204건
최종 데이터: 24280건

=== 제거된 불일치 데이터 샘플 (상위 10건) ===
            title province gu_name                      addr1        mapx       mapy
66        난정리 전망대       인천     강화군      인천광역시 강화군 교동면 난정리 729  126.229495  37.776280
70   러블리펫108 애견펜션       인천     옹진군    인천광역시 옹진군 영흥면 선재로 76-27  126.530918  37.236529
73          루시아펜션       인천     옹진군   인천광역시 옹진군 백령면 백령로 461-18  124.697673  37.973404
76            뻘다방       인천     옹진군       인천광역시 옹진군 선재로 55 뻘다방  126.531288  37.234240
87     옹진 백령도 두무진       인천     옹진군   인천광역시 옹진군 백령면 연화리 1026-9  124.619455  37.974266
450  옥천군 반려동물 놀이터       충북     옥천군           충청북도 옥천군 이원면 장찬리  127.598018  36.249775
452       월류봉 둘레길       충북     영동군           충청북도 영동군 황간면 원촌리  127.891718  36.234316
541       해오름관광펜션       경북     울릉군    경상북도 울릉군 울릉읍 도동1길 35-10  130.906443  37.483143
545        거창 수승대       경남     거창군        경상남도 거창군 위천면 은하리길 2  127.833697  35.760966
577    경천애인 징검다리길       전북     완주군  전북특별자치도 완주군 경천면 경천리 680-3  127.251789  36.02100

In [35]:

# 필터링 후 CSV 저장
OUTPUT_DIR = "./data"
csv_path = f"{OUTPUT_DIR}/pet_tour_preprocessed.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"✅ 저장 완료: {csv_path}")
print(f"   최종 장소 수: {len(df)}건 (불일치 204건 제거)")


✅ 저장 완료: ./data/pet_tour_preprocessed.csv
   최종 장소 수: 24280건 (불일치 204건 제거)
